In [17]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from pathlib import Path
import spacy as sp

In [18]:
input_path=Path().resolve()/'input'

# Loading Dataa

In [19]:
testing_data=pl.read_csv(input_path/'Corona_NLP_test.csv')

In [20]:
training_data=pl.read_csv(input_path/'Corona_NLP_train.csv')

# Data Preprocessing

### Data Preprocessing pipeline:
1. clean missing values - Done
2. fix data types -Done
3. Text Cleaning -Done
4. tokenize the text of the target column -Done
5. remove stop words Done
6. Stemming words (Porter Stemmer) -Done

Note: Lemmatization is a very costly operation that takes minutes sometimes hours which is not suitable entirely for this project

In [21]:
testing_data.null_count()

UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
u32,u32,u32,u32,u32,u32
0,0,834,0,0,0


In [22]:
# checking nulls
training_data.null_count()

UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
u32,u32,u32,u32,u32,u32
0,0,8590,0,0,0


null exists at location

We will impute those nulls with unknown value

In [23]:
training_data=training_data.with_columns(pl.col('Location').fill_null('unknown'))
training_data.null_count()

UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


In [24]:
testing_data=testing_data.with_columns(pl.col('Location').fill_null('unknown'))
testing_data.null_count()

UserName,ScreenName,Location,TweetAt,OriginalTweet,Sentiment
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


In [25]:
# Converting the tweetat to polars datetime
training_data=training_data.with_columns(pl.col('TweetAt').str.to_date())
testing_data=testing_data.with_columns(pl.col('TweetAt').str.to_date())

#### clean data and tokenize

In [26]:
from lib import TextClassification
preprocessor=TextClassification(training_data)

In [27]:
# Method one using spacy cleaner --> very slow
# data=training_data.with_columns(pl.col('OriginalTweet').map_elements(lambda x: preprocess_spacy(x), return_dtype=pl.String))
# data.select(pl.col('OriginalTweet')).to_series().to_list()

In [28]:
# Very Fast method --polars nativen method
training_data=training_data.pipe(preprocessor.preprocess_pl_native)
testing_data=testing_data.pipe(preprocessor.preprocess_pl_native)

#### Removing stop words and punc

In [29]:
training_data=training_data.with_columns(
    pl.col('OriginalTweet').str.split(' ').map_elements(
        lambda x: preprocessor.remove_stop_words_and_punc(x), return_dtype=pl.String))
testing_data=testing_data.with_columns(
    pl.col('OriginalTweet').str.split(' ').map_elements(
        lambda x: preprocessor.remove_stop_words_and_punc(x), return_dtype=pl.String))

### Stemming

In [30]:
# Stemming words using porter stemming
training_data=training_data.with_columns(
    pl.col('OriginalTweet').map_elements(
        lambda x: preprocessor.porter_stem(x), return_dtype=pl.String))

testing_data=testing_data.with_columns(
    pl.col('OriginalTweet').map_elements(
        lambda x: preprocessor.porter_stem(x), return_dtype=pl.String))

# Data Exploration

### Data Exploration Steps:
1. get insights from the data
2. printing statistics
3. Checking missing values and duplicates
4. checking categorical distribution
5. Printing some plots (Barplot, Pie chart)
6. Word Cloud Plot
7. Word Cloud For each sentiment
8. Word counter for each sentiment
9. Text Length for each Sentiment

In [ ]:
training_data.head(10)

In [ ]:
training_data.describe()

### Checking duplicates


In [ ]:
is_dup=training_data.is_duplicated()
training_data.filter(is_dup)

There are no duplicates within the data

There are many null values in the location field <br>
Date format for the TweetAt is not accurate needs to be changed

In [ ]:
## understanding screen name
training_data.select(pl.col('ScreenName')).unique().count()

In [ ]:
# All values are unique so its an identity to the test

In [ ]:
# describing the categorical variables
training_data.to_pandas().describe(include="object")

- Most common location is Londan 
- Most common tweet date is 3448
- Most common Sentiment is Positive

In [ ]:
# Visualizing the sentiment column`

In [ ]:
from lib import TextClassification
sentiment_data=training_data.group_by('Sentiment').len().select(pl.col('Sentiment'), pl.col('len').alias('Counts'))

In [ ]:
Plots=TextClassification(sentiment_data)
Plots.barplot_seaborn('Sentiment','Counts')

In [ ]:
Plots.pie('Counts','Sentiment')

In [ ]:
# Correlation between Sentiment and Original Tweet
data_corr=training_data.select(pl.corr('OriginalTweet','Sentiment',method="spearman"))
data_corr

### Word cloud plot

In [ ]:
Plots=TextClassification(training_data)
Plots.WorldCloud()

### Word cloud for Neutral Sentiment

In [ ]:
Plots.WorldCloud('Neutral')

### Word cloud for Extermely Negative Sentiment

In [ ]:
Plots.WorldCloud('Extremely Negative')

### Word cloud for Negative Sentiment

In [ ]:
Plots.WorldCloud('Negative')

### Word cloud for Positive Sentiment

In [ ]:
Plots.WorldCloud('Positive')

### Word cloud for Exteremly Positive Sentiment

In [ ]:
Plots.WorldCloud('Extremely Positive')

_______________________

### Top Words Count for Neutral Sentiment

In [ ]:
Plots.Top_words('Neutral')

### Top Words Count for Extermely Negative Sentiment

In [ ]:
Plots.Top_words('Extremely Negative')

### Top Words Count for Negative Sentiment

In [ ]:
Plots.Top_words('Negative')

### Top Words Count for Positive Sentiment

In [ ]:
Plots.Top_words('Positive')

### Top Words Count for Exteremly Positive Sentiment

In [ ]:
Plots.Top_words('Extremely Positive')

### Boxplots for the Text Length for each Sentiment

In [ ]:
Plots.boxplot()

# Modelling

### Modelling Pipeline:
1. Apply TFIDF vectorizer to the training and testing data
2. Run the three selected models (Logistic Regression, Naive Bayes, Random Forest)
3. Create a function to evaluate those models with visualization
4. compare the performance of those models.
5. Perform Hyperparameter tuning -> a function applied to all three models
6. Compare model's performance before and after tuning with visualization
7. Selection of the best Model based on the evaluation metrics. 

In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf_idf=TfidfVectorizer()
tf_idf?

Type:        TfidfVectorizer
String form: TfidfVectorizer()
File:        ~/.virtualenvs/crosscompute/lib/python3.11/site-packages/sklearn/feature_extraction/text.py
Docstring:  
Convert a collection of raw documents to a matrix of TF-IDF features.

Equivalent to :class:`CountVectorizer` followed by
:class:`TfidfTransformer`.

For an example of usage, see
:ref:`sphx_glr_auto_examples_text_plot_document_classification_20newsgroups.py`.

For an efficiency comparison of the different feature extractors, see
:ref:`sphx_glr_auto_examples_text_plot_hashing_vs_dict_vectorizer.py`.

For an example of document clustering and comparison with
:class:`~sklearn.feature_extraction.text.HashingVectorizer`, see
:ref:`sphx_glr_auto_examples_text_plot_document_clustering.py`.

Read more in the :ref:`User Guide <text_feature_extraction>`.

Parameters
----------
input : {'filename', 'file', 'content'}, default='content'
    - If `'filename'`, the sequence passed as an argument to fit is
      expected to b